# 2-Chip Placement Optimization

Finds optimal chip positions on a heat sink to minimize weighted thermal resistance.

**Prerequisites:** py2femm server running on `localhost:8082`

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path
repo_root = Path.cwd().resolve()
while repo_root.name and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

example_dir = str(repo_root / "examples" / "heatflow" / "heatsink")
if example_dir not in sys.path:
    sys.path.insert(0, example_dir)

import heatsink_optimize as opt
from py2femm.client import FemmClient

import matplotlib
%matplotlib inline

assert opt.server_is_healthy(), "py2femm server not running on localhost:8082"
print("Server OK")

cfg = opt.OptimConfig(grid_n=5, max_iter=15, timeout=60)
client = FemmClient(mode="remote", url="http://localhost:8082")
print(f"Base: {cfg.heatsink.base_w}x{cfg.heatsink.base_h} mm")
print(f"ChipA: {cfg.chip_a.power}W, ChipB: {cfg.chip_b.power}W")

In [ ]:
grid_results = opt.brute_force(cfg, client)
print(f"\n{len(grid_results)} feasible points evaluated")

In [ ]:
scipy_result = None
if grid_results:
    best_grid = min(grid_results, key=lambda r: r["objective"])
    scipy_result = opt.scipy_optimize(cfg, client, x0=(best_grid["x_a"], best_grid["x_b"]))

In [ ]:
opt.plot_grid_results(cfg, grid_results, scipy_result)